# CPT Notebook A — Chuẩn bị data (Section 6.1)

**Chạy đúng 1 lần, không cần GPU.** Output: dataset đã tokenize + pack 2048 token tại `{HF_USER}/vi-cpt-corpus-2048` (private) — Notebook B (`cpt_b_train.ipynb`) sẽ load thẳng repo này, không phải tokenize lại mỗi session.

Nguồn data (đều public, không gated, parquet native — không dùng CC100/OSCAR vì là dataset kiểu script/gated, `datasets` >= 3.0 không load được):
- `wikimedia/wikipedia` config `20231101.vi` — chất lượng cao
- `HuggingFaceFW/fineweb-2` config `vie_Latn` — web corpus tiếng Việt đã lọc chất lượng, quy mô lớn (stream)
- `HuggingFaceFW/fineweb-edu` `sample-10BT` — English replay ~20% chống catastrophic forgetting

**Trước khi chạy:** Add-ons → Secrets → thêm `HF_TOKEN` (write token); sửa `HF_USER` ở cell Config.

In [ ]:
!pip install -q -U datasets huggingface_hub

In [ ]:
# Config + login
HF_USER = "<user>"  # TODO: đổi thành username Hugging Face của bạn
DATA_REPO = f"{HF_USER}/vi-cpt-corpus-2048"
BLOCK = 2048

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# Tải 3 nguồn — mục tiêu tổng ~400M token (đủ cho max_steps=6000, tỉ lệ vi:en ≈ 80:20)
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets

wiki_vi = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train")  # ~1.3M bài

web_vi = load_dataset("HuggingFaceFW/fineweb-2", "vie_Latn", split="train", streaming=True)
web_vi = Dataset.from_list([                     # stream, chỉ giữ cột text, lấy 1.5M đoạn
    {"text": r["text"]} for _, r in zip(range(1_500_000),
        (r for r in web_vi if len(r["text"]) > 200))
])

fineweb_en = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT",
                          split="train", streaming=True)
fineweb_en = Dataset.from_list([{"text": r["text"]} for _, r in zip(range(150_000), fineweb_en)])
print(f"wiki_vi={len(wiki_vi)}  web_vi={len(web_vi)}  fineweb_en={len(fineweb_en)}")

In [ ]:
# Tokenize + pack thành block 2048 token (chuẩn CPT: nối tài liệu bằng EOS rồi cắt khúc)
from transformers import AutoTokenizer
from itertools import chain
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B-Base")

def pack(ds):
    def tokenize(batch):
        return {"ids": [tok(t)["input_ids"] + [tok.eos_token_id] for t in batch["text"]]}
    def group(batch):
        flat = list(chain.from_iterable(batch["ids"]))
        n = (len(flat) // BLOCK) * BLOCK
        return {"input_ids": [flat[i:i+BLOCK] for i in range(0, n, BLOCK)]}
    ds = ds.select_columns(["text"]).map(tokenize, batched=True, remove_columns=["text"], num_proc=4)
    return ds.map(group, batched=True, batch_size=1000, remove_columns=["ids"], num_proc=4)

vi = pack(concatenate_datasets([wiki_vi.select_columns(["text"]),
                                web_vi])).shuffle(seed=42)
en = pack(fineweb_en).shuffle(seed=42)
print(f"packed: vi={len(vi)} block  en={len(en)} block  ({BLOCK} token/block)")

In [ ]:
# Trộn 80/20, tách eval_vi + eval_en (theo dõi catastrophic forgetting), push lên Hub
n_en = min(len(en), int(len(vi) * 0.25))  # 0.25 × vi ≈ 20% tổng
train = concatenate_datasets([vi.select(range(len(vi) - 1000)),
                              en.select(range(n_en - 1000))]).shuffle(seed=42)
DatasetDict({
    "train":   train,
    "eval_vi": vi.select(range(len(vi) - 1000, len(vi))),
    "eval_en": en.select(range(n_en - 1000, n_en)),
}).push_to_hub(DATA_REPO, private=True)
print(f"Xong — data nằm ở {DATA_REPO}. Chuyển sang cpt_b_train.ipynb.")